# 地図の座標系入門
緯度経度で保存されたPOIを、距離・面積・密度へ変換します。吉祥寺駅の固定代表点と保存済みPOIを使い、既存の分析結果は変更しません。
位置を保存する座標、Webで表示する座標、計算する座標は分けて選びます。EPSGは座標参照系などを識別する番号です。
[方法と再実行](../docs/crs.md) / [検証](../docs/crs-validation.md) / [記事資料](../docs/crs-article-handoff.md)
準備：POIテーマで `uv sync --locked --all-groups`。固定POIとNatural Earth 5.1.1がローカルに必要です。本Notebookはネットワーク取得しません。

In [ ]:
from geoai_open_lab.crs_experiment import run, OUT, station, transformer
from geoai_open_lab.crs_figures import render
from IPython.display import display, SVG
tables = run()
render()
display(SVG(filename=str(OUT / 'overview.svg')))

## 1. 座標を変換する
`always_xy=True`なら入力は経度・緯度、平面座標の出力は東・北です。EPSG:4326の公式軸順（緯度・経度）や、日本の平面直角座標のX＝北・Y＝東とは区別します。
往復が一致しても、元の位置が同じ精度で正しいとは限りません。今回の基準系変換の公称精度は1mです。

In [ ]:
s = station()
lon, lat = s['longitude'], s['latitude']
x, y = transformer(6677).transform(lon, lat)
print('東・北 [m]:', x, y)
print('逆変換:', transformer(4326, 6677).transform(x, y))
print('Web表示用:', transformer(3857).transform(lon, lat))

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
point = gpd.GeoSeries([Point(lon, lat)])
labeled = point.set_crs(4326)  # 数値は変わらない。既知の元CRSを指定
projected = labeled.to_crs(6677)  # 数値を変換
print(labeled.iloc[0], '→', projected.iloc[0])
try:
    transformer(3857).transform(lat, lon, errcheck=True)
except Exception as error:
    print('経度・緯度を逆にした例:', type(error).__name__)

## 2. 同じ距離を計算して比べる
駅から方位と測地線距離で作った説明用の地点です。実在施設の名前ではありません。測地線距離を基準に、平面上の差をメートルと%で確認します。`raw_degree_distance`は誤った緯度経度のユークリッド計算で、単位は度のままです。メートルとの差として解釈しません。

In [ ]:
display(tables['distances'].round(6))
display(SVG(filename=str(OUT / 'coordinates.svg')))

## 3. POIを500mで抽出する
測地線、東京の平面直角座標、Webメルカトルを比較します。ポリゴンの円近似と正確な平面距離で採用点が一致することも内部で確認します。POI登録数であり営業中店舗数ではありません。

In [ ]:
display(tables['poi_summary'])
display(tables['poi_overlap'])
display(SVG(filename=str(OUT / 'poi_500m.svg')))

## 4. 世界の面積と密度
Natural Earthの同じ境界を使います。図は南極を除外し、日付変更線で分かれた元のポリゴンを保持します。面積は粗い境界の測地面積との比較で、公式国土面積ではありません。
密度例は**合成データ**です。赤道・東京・北緯65度に、同じ測地面積100km²とPOI数1,000件を置きます。イコールアースは面積を保ちますが、形・距離・角度をすべて保つ図法ではありません。

In [ ]:
display(tables['world_areas'].round(6))
display(SVG(filename=str(OUT / 'world_projections.svg')))
display(tables['synthetic_density'].round(6))
display(SVG(filename=str(OUT / 'features_flow.svg')))

## 5. 地域特徴量・GeoAIへ
既存656駅の代表点は6677の適用範囲内で、投影縮尺による局所距離差は0.01%以内でした。今回の東京の小範囲分析には妥当として、既存計算は変更しません。
AOI・密度・集中度には投影法が影響します。TF-IDF、SVD、K-meansは座標系を直接扱いませんが、入力となるPOI集合や特徴量に依存します。世界の分析へ東京の座標系を流用しません。
許容誤差は用途ごとに決めます。本実験は水平位置の導入であり、測量・地籍の精度保証や道路到達圏の検証ではありません。

In [ ]:
display(tables['station_audit'][['meridional_scale', 'parallel_scale', 'areal_scale']].agg(['min', 'max']))
display(SVG(filename=str(OUT / 'selection.svg')))